# PointDiT in motion: Kaggle evaluation

CPU only (Accelerator: None), Internet on. Attach the "PointDiT TUM RGB-D Subset" dataset and the output of the runner notebook version that wrote the predictions (it holds `predictions/<run name>/run.json`). Add the Kaggle secret `GITHUB_TOKEN` while the repo is private. No GPU, DINOv3, weights or `HF_TOKEN` are needed.

With `RUN_EVALUATION = False` (default) the notebook does not re-evaluate: also attach the output of the evaluation notebook version that wrote `eval/<run name>/per_frame.csv`, and it reuses that CSV. With `RUN_EVALUATION = True` it runs `evaluate.py` (including `rel_d_near/mid/far` and `focal_ratio`) into `/kaggle/working/eval/<run name>/`; if an older `per_frame.csv` of the same run is attached, `compare_eval_csv.py` then checks that every shared column is reproduced within 1e-6 and stops the notebook otherwise. The preview step writes RGB | predicted depth | GT depth | relative error panels and `error_breakdown.csv` to `/kaggle/working/previews/<run name>/`.

With `RUN_FOCAL_SPLIT = True` (independent of `RUN_EVALUATION`) it runs `focal_reliability.py`, which refits the implied focal length on all pixels and on the left and right image sides, checks that the full fit reproduces `per_frame.csv`'s `focal_ratio` within 1e-6, and writes `/kaggle/working/eval/<run name>/focal_split.csv` (a few minutes).

With `RUN_CONSISTENCY = True` (independent of the other flags) it runs `consistency_eval.py` on the same predictions and dataset, right after locating them, with 4 workers (one sequence each, 1 thread, ~4 GB RAM each) and per-pixel M2 maps for fr1_360 and fr1_desk at 33 ms, into `/kaggle/working/consistency/<run name>/` (about 2–3 h). It ends with a regression check against the subtask-1 development values and stops the notebook if any differs by more than 1e-3. Rerunning in the same session skips sequences whose `pairs_<sequence>.csv` is already complete.

With `RUN_PERSISTENCE = True` (independent of the other single-run flags) it runs `persistence_eval.py` on the same predictions and dataset, with 4 workers (one sequence each, 1 thread), into `/kaggle/working/persistence/` (about 1–1.5 h). Per pair it compares frame i's GT-aligned prediction warped into j with frame j's own prediction against GT_j, and ends with a regression check: every pair's median |E_i − E_j| must equal `m1_visible_median` (and `m1_interior_median`) of the bundled `results/consistency/baseline/pairs_<sequence>.csv` within 1e-6, with identical pair keys; otherwise the notebook stops. Rerunning in the same session skips sequences whose `persistence_<sequence>.csv` is already complete.

With `RUN_FUSION = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it evaluates temporal fusion at **ORACLE scale**, an upper bound in which every frame's prediction is first aligned to its own GT. It needs the same inputs as the other runs, namely the dataset, the runner output with the predictions and the evaluation output with `per_frame.csv` (or `RUN_EVALUATION = True`). `fusion_prepare.py` caches every window frame's GT-aligned depth in `/kaggle/working/fusion/oracle_aligned/<run name>/` (about 1.5 GB, 4 workers, 1 thread each), then `fusion_eval.py` fuses 100 targets per sequence for every configuration with 4 workers of 1 thread each and writes `oracle_fusion_<sequence>.csv`, `oracle_fusion_summary.csv` and `oracle_fusion.json` to `/kaggle/working/fusion/oracle_eval/<run name>/` (about 2 to 3 h in all). A preflight first scores `baseline_self` on the first 3 targets of each sequence and stops the notebook if any differs from `per_frame.csv` by more than 1e-4 (about a minute). The full run stops the notebook if the cached aligned depth differs from `per_frame.csv` by more than 1e-6. It writes every output first, then checks every `baseline_self` row at 1e-4 and writes `regression_status.json`; a mismatch there only prints a warning, and the cell prints `regression_status.json` either way. Set `FUSION_SEQUENCES` to a list of sequence names to evaluate only those (default None means all). Complete sequence CSVs already in the output folder are skipped, and the summary and json cover every sequence CSV there. Set `FUSION_CONFIG_GROUP = "edge"` to run the edge-aware configurations instead of the ladder, written to `oracle_eval_edge/<run name>/` (about 1 h). Set `FUSION_CACHE_INPUT` to a glob pattern matching exactly one attached output's `oracle_aligned` folder to copy that cache into `/kaggle/working/fusion/` and skip the alignment hour. Set `FUSION_ALL_TARGETS_CONFIG` to a configuration name to also run that one on all 300 targets per sequence into `oracle_eval_all_targets/<run name>/`. Rerunning in the same session skips cached frames and complete sequences.

With `RUN_FUSION_JOINT = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it runs `fusion_joint.py`, which removes GT from the scale step. For every target of `fusion_eval.py` and the bases `causal_adjacent_w4`, `causal_diverse_w4` and `centred_adjacent_w4` (median aggregator), each window frame's depth scale and shift are solved jointly from parallax under the metric GT poses, which stay an **ORACLE**, then the solved depths are fused and only the fused output is aligned to the target's GT before scoring, as `baseline_self` is. Every output is marked `scale = joint_solved` and `poses = oracle_gt`. It reads the GT-aligned depth cache for a logging-only comparison with the oracle affine, from `/kaggle/working/fusion/oracle_aligned/<run name>/` when `RUN_FUSION` is also True and otherwise from the attached output that `FUSION_CACHE_INPUT` matches, without copying it. A preflight first scores `baseline_self` on 3 targets per sequence against `per_frame.csv` at 1e-4 and runs the solver on one target per sequence, and stops the notebook on a failure. The run (4 workers, 1 thread each, about 1.5 to 2 h) writes `joint_fusion_<sequence>.csv`, `joint_fusion_summary.csv`, `joint_fusion.json` and `regression_status.json` to `/kaggle/working/fusion/joint_eval/<run name>/`, then checks that `baseline_self` reproduces `per_frame.csv` and that the no-fusion control reproduces `baseline_self`, both at 1e-4; a mismatch only prints a warning. The summary also lists the oracle scale result of each base from the edge oracle summary, which was not realigned after fusion.

With `RUN_RENDER_PREPARE = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it runs `fusion_render_prepare.py`, which saves fused depth for videos and 3D maps. Every window frame of the `RENDER_SEQUENCES` (default fr1_360 and fr1_desk, None means every sequence) that has four earlier source frames by time runs exactly the `joint_causal_adjacent_w4` pipeline of `fusion_joint.py`, so `scale = joint_solved` and `poses = oracle_gt` (an **ORACLE**). Frames with fewer sources keep the raw depth only and have `has_fused` false. Each frame is written as a float16 `<index>.npz` holding `raw_aligned` and `fused_aligned` (both after the per frame alignment to GT that `baseline_self` is scored on), `rel_d_raw`, `rel_d_fused`, `low_parallax`, `n_sources`, `has_fused`, `sequence`, `index` and `timestamp`, under `/kaggle/working/fusion/render/<run name>/<sequence>/`, with `render_prepare.json` and `regression_status.json` beside them (4 workers, 1 thread each, about 370 MB per sequence). It reads the aligned depth cache only for provenance, from this session when `RUN_FUSION` is True and otherwise from `FUSION_CACHE_INPUT`, which may match the attached notebook output root or its `fusion/oracle_aligned/<run name>` folder directly (the joint cell accepts both too). A preflight first checks 3 targets per sequence and stops the notebook on a failure. After every frame is written, the targets at window positions 100 to 199 must reproduce `per_frame.csv` at 1e-4 in `rel_d_raw` and the joint results at 1e-3 in `rel_d_fused`, read from this session's `joint_eval/<run name>/` when `RUN_FUSION_JOINT` wrote it and otherwise from the bundled `results/fusion/joint_eval/<run name>/`; a mismatch only prints a warning. Rerunning skips frames already written.

With `RUN_RENDER_VIDEO = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it renders the render preparation output into `/kaggle/working/videos/`, labelled on screen as oracle poses, an upper bound, with joint solved scale. The input is this session's `RUN_RENDER_PREPARE` output when `RENDER_INPUT` is None, otherwise the one path `RENDER_INPUT` matches, either an attached notebook output root holding `fusion/render/<run name>/` or that folder itself. For each of the `RENDER_SEQUENCES` (None means every sequence folder) `render_video.py` writes a 1920x1080 H.264 mp4 of RGB, raw, fused and GT depth and the raw and fused temporal disagreement with the previous frame, followed by a slow motion replay of the 2 seconds where fusion helps most, plus a 12 second half size gif, a poster png and a json of settings and statistics. For the first sequence `export_fused_map.py` writes GT, raw and fused point maps from every 5th fused frame as binary PLY with a ghosting statistic, and `render_orbit.py` renders a 12 second orbit of the three side by side as mp4 and gif (`RENDER_FLYTHROUGH = True` adds a fused map flythrough). Without ffmpeg the videos are PNG frame folders and the scripts print the encode command.

With `RENDER_STORY = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it renders short, slower clips that are easier to read than the dashboard, from the same input as `RUN_RENDER_VIDEO` (this session's render preparation output or `RENDER_INPUT`), into `/kaggle/working/videos/story/`. For each of the `RENDER_SEQUENCES` `render_story.py` runs every mode in `STORY_MODES`, by default `hero` (RGB, raw and fused depth on one fixed range, first and last frame held 1 s), `flicker` (raw and fused temporal disagreement over the dimmed RGB with live medians), `toggle` (raw and fused depth alternating every 0.5 s), `stillflip` (one frame flipping between raw and fused, plus a poster), `cloud` (one frame as raw, fused and GT point clouds from a swaying camera) and `cloud_moving` (the clip as point clouds from a camera turned 10 degrees). All run at 10 fps and write a 1920x1080 mp4, a half size gif and a json of settings, frame choice and clip medians. `STORY_ARGS` adds per mode options such as `--start` and `--max-frames`. For the first sequence it then renders a slowed orbit of the raw, fused and GT point maps side by side (36 s at 10 fps, ends held 1 s), exporting the point maps first unless `RUN_RENDER_VIDEO` already did. Labelled as oracle poses, an upper bound, with joint solved scale.

With `RENDER_WORLD = True` (independent of the other single-run flags, never together with `RUN_SAMPLING_EVAL`) it renders world frame videos of the raw and fused depth side by side from the same input as `RUN_RENDER_VIDEO` (this session's render preparation output or `RENDER_INPUT`), into `/kaggle/working/videos/world/`. For each of the `RENDER_SEQUENCES` `render_world_video.py` runs every mode in `WORLD_MODES`, by default `accumulate` (every 3rd fused frame adds its points in the world frame, so the map grows, at most 400000 points per source with the oldest thinned) and `sliding` (only the points of the last 30 dataset frames). One virtual camera on a slow fixed orbit, framed from the whole clip, views every panel, and a small yellow frustum marks the current real camera. Each mode writes a 1920x1080 mp4 at 10 fps, a half size gif and a json of settings, point counts and the ghosting statistic of the final accumulated map per source (about 4 to 6 min per mode and sequence, under 1 GB of RAM). `WORLD_ARGS` adds options to every mode, such as `--with-gt` for a third GT panel. Labelled as oracle GT poses, an upper bound, joint solved scale, not dynamic 4D reconstruction.

Discovery of `run.json` and `per_frame.csv` under `/kaggle/input` ignores every path that contains `/Pointdit-In-Motion/`, so a project checkout inside an attached output is never picked up. Set `PER_FRAME_CSV_OVERRIDE` to the exact path of a `per_frame.csv` to use that file instead of discovering one.

With `RUN_SAMPLING_EVAL = True` (all other flags False) only the sampler-initialisation comparison runs; the single-run steps are skipped. Attach the runner output that holds `predictions/{noise_seed0,noise_seed1,noise_seed2,zeros_rerun}` and the runner output that holds `predictions/baseline`; each run is found by globbing for its `run.json`. It repeats the determinism check (zeros_rerun vs baseline), then runs `evaluate.py` (4 workers, 1 thread each) and `consistency_eval.py` (two configs at a time, one worker per sequence, 1 thread) for the four configs on `configs/sampling_subsets.yaml`. Finally `sampling_compare.py` writes everything to `/kaggle/working/sampling/` and stops the notebook if the zero-init pairs differ from `results/consistency/baseline` by more than 1e-3 (about 3 h in all).

With `RUN_SWEEP_EVAL = True` (independent of the other flags) it evaluates the compute sweep. Attach the runner output that holds `predictions/sweep/<run name>/`; each run is found by its `run.json` (paths containing `/Pointdit-In-Motion/` are ignored), and only runs whose `sweep_status.json` says `ok` are evaluated. It runs `evaluate.py` (4 workers) on `configs/sweep_subsets.yaml` for each into `/kaggle/working/sweep/eval/<run name>/`, then `sweep_accuracy.py` writes `/kaggle/working/sweep/sweep_accuracy.csv` with per config the median per-frame `rel_d`, the median per-frame difference against `sweep_baseline_check` and a moving-block bootstrap interval (block 15). Last, it checks `sweep_baseline_check`'s `rel_d` against the bundled `results/eval/baseline/per_frame.csv` on the same frames within 1e-3; a mismatch prints a banner and is a warning, not an error. The single-run steps are skipped unless `PREDICTIONS_DIR` is set (about 2 h).

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
PROJECT_BRANCH = "main"
# None: use the single run.json found under /kaggle/input; set a path if several are attached.
PREDICTIONS_DIR = None
# False: reuse the single per_frame.csv found under /kaggle/input instead of running evaluate.py (~1.5 h).
# True: run evaluate.py, then compare with the per_frame.csv attached under /kaggle/input, if any.
RUN_EVALUATION = False
# True: refit the implied focal length per image side and write focal_split.csv next to the evaluation.
RUN_FOCAL_SPLIT = False
# True: run consistency_eval.py (all frame pairs at 33/100/333 ms) into /kaggle/working/consistency/<run name>/.
RUN_CONSISTENCY = False
# True: run persistence_eval.py (error persistence over the same pairs) into /kaggle/working/persistence/.
RUN_PERSISTENCE = False
# True: run fusion_prepare.py and fusion_eval.py at ORACLE scale (an upper bound) into /kaggle/working/fusion/.
RUN_FUSION = False
# True: run fusion_joint.py, fusion with jointly solved per frame scale and ORACLE GT poses, into
# /kaggle/working/fusion/joint_eval/<run name>/. Needs the aligned depth cache (RUN_FUSION or FUSION_CACHE_INPUT).
RUN_FUSION_JOINT = False
# True: run fusion_render_prepare.py, the raw and fused depth of every window frame by joint_causal_adjacent_w4, into
# /kaggle/working/fusion/render/<run name>/. Needs the aligned depth cache (RUN_FUSION or FUSION_CACHE_INPUT).
RUN_RENDER_PREPARE = False
# Sequences of RUN_RENDER_PREPARE; None renders every sequence of the cache.
RENDER_SEQUENCES = ["rgbd_dataset_freiburg1_360", "rgbd_dataset_freiburg1_desk"]
# True: run render_video.py on RENDER_SEQUENCES, then export_fused_map.py and render_orbit.py on the first one, into
# /kaggle/working/videos/. Reads the render preparation output of this session or of RENDER_INPUT.
RUN_RENDER_VIDEO = False
# None uses this session's RUN_RENDER_PREPARE output. Otherwise a path or glob matching one attached notebook output
# root (holding fusion/render/<run name>/) or the folder that holds the sequence folders of npz frames.
RENDER_INPUT = None
# True: also render the fused map flythrough along the recorded camera path.
RENDER_FLYTHROUGH = False
# True: run render_story.py on RENDER_SEQUENCES and a slowed side by side orbit of the first one into
# /kaggle/working/videos/story/. Reads the same input as RUN_RENDER_VIDEO.
RENDER_STORY = False
STORY_MODES = ["hero", "flicker", "toggle", "stillflip", "cloud", "cloud_moving"]
# Extra render_story.py options per mode; every mode takes --fps, --start and --max-frames.
STORY_ARGS = {"toggle": ["--start", 20, "--max-frames", 80], "cloud_moving": ["--start", 20, "--max-frames", 120]}
# True runs render_world_video.py on RENDER_SEQUENCES, world frame point maps of raw and fused depth from a slow
# orbit, into /kaggle/working/videos/world/. Reads the same input as RUN_RENDER_VIDEO.
RENDER_WORLD = False
WORLD_MODES = ["accumulate", "sliding"]
# Extra render_world_video.py options for every mode, for example ["--with-gt"] for a third GT panel.
WORLD_ARGS = []
# None discovers the single per_frame.csv under /kaggle/input; set an exact path to use that file instead.
PER_FRAME_CSV_OVERRIDE = None
# A fusion configuration name to rerun on all 300 targets per sequence after the main fusion run, or None.
FUSION_ALL_TARGETS_CONFIG = None
# Each worker needs ~3.5 GB of RAM; torch already uses every core for one frame.
WORKERS = 1
# True: only compare sampler initialisations (noise_seed0-2, zeros_rerun) on configs/sampling_subsets.yaml into
# /kaggle/working/sampling/; needs every other flag False.
RUN_SAMPLING_EVAL = False
if RUN_SAMPLING_EVAL and (RUN_EVALUATION or RUN_FOCAL_SPLIT or RUN_CONSISTENCY or RUN_PERSISTENCE or RUN_FUSION
                          or RUN_FUSION_JOINT or RUN_RENDER_PREPARE or RUN_RENDER_VIDEO or RENDER_STORY
                          or RENDER_WORLD):
    raise ValueError("RUN_SAMPLING_EVAL runs on its own: set RUN_EVALUATION, RUN_FOCAL_SPLIT, RUN_CONSISTENCY, "
                     "RUN_PERSISTENCE, RUN_FUSION, RUN_FUSION_JOINT, RUN_RENDER_PREPARE, RUN_RENDER_VIDEO, "
                     "RENDER_STORY and RENDER_WORLD to False")
if FUSION_ALL_TARGETS_CONFIG and not RUN_FUSION:
    raise ValueError("FUSION_ALL_TARGETS_CONFIG needs RUN_FUSION = True")
# True evaluates every successful compute sweep run on configs/sweep_subsets.yaml and write
# /kaggle/working/sweep/sweep_accuracy.csv. Independent of the other flags; the single-run steps then need
# PREDICTIONS_DIR, since the sweep attaches several run.json.
RUN_SWEEP_EVAL = False
SINGLE_RUN = not RUN_SAMPLING_EVAL and (not RUN_SWEEP_EVAL or PREDICTIONS_DIR is not None)
if RUN_SWEEP_EVAL and not SINGLE_RUN and (RUN_EVALUATION or RUN_FOCAL_SPLIT or RUN_CONSISTENCY
                                          or RUN_PERSISTENCE or RUN_FUSION or RUN_FUSION_JOINT
                                          or RUN_RENDER_PREPARE or RUN_RENDER_VIDEO or RENDER_STORY
                                          or RENDER_WORLD):
    raise ValueError("with RUN_SWEEP_EVAL the single-run flags need PREDICTIONS_DIR set to the run to evaluate")

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"

import base64
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Only needed while the repo is private. Git reads the token from the environment, so it
# never appears in a command, in cell output, or in the clone's .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

if os.path.isdir(f"{PROJECT_DIR}/.git"):
    !git -C {PROJECT_DIR} fetch -q origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} checkout -q {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull -q --ff-only origin {PROJECT_BRANCH}
else:
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"


def run(*cmd: object, allowed: tuple[int, ...] = (0,)) -> int:
    """Run a project script, streaming its output, and fail the cell on an exit code outside allowed."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() not in allowed:
        raise RuntimeError(f"{cmd[0]} exited with code {proc.returncode}; see the output above")
    return proc.returncode

In [ ]:
!pip install -q "numpy>=1.26.0" "Pillow>=10.0.0" "matplotlib>=3.9.0" "PyYAML>=6.0" "torch>=2.4"

import torch

print("torch", torch.__version__, "| CPUs", os.cpu_count(), "| CUDA", torch.cuda.is_available())

In [ ]:
import csv
import json
from pathlib import Path

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences
from pim.runs import IGNORED_INPUT_PARTS
from pim.runs import find_input_files as find_files

INPUT = Path("/kaggle/input")


def find_input_files(name: str) -> list[Path]:
    """Every file called `name` under /kaggle/input, following symlinked folders and ignoring any path that contains
    one of IGNORED_INPUT_PARTS, such as a project checkout bundled into an attached output."""
    return find_files(INPUT, name, IGNORED_INPUT_PARTS)


def input_overview(max_depth: int = 4) -> str:
    lines = []
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        depth = len(Path(root).relative_to(INPUT).parts)
        if depth >= max_depth:
            dirs.clear()
        dirs[:] = [d for d in sorted(dirs) if d not in ("rgb", "depth")]
        lines.append(f"{'  ' * depth}{Path(root).name}/  ({len(files)} files)")
    return "\n".join(lines)


sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)
data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()

SAMPLING_RUN_NAMES = ("noise_seed0", "noise_seed1", "noise_seed2", "zeros_rerun", "baseline")


def find_runs(names: tuple[str, ...]) -> dict[str, Path]:
    by_name: dict[str, list[Path]] = {}
    for path in find_input_files("run.json"):
        by_name.setdefault(json.loads(path.read_text())["name"], []).append(path.parent)
    wrong = {name: [str(p) for p in by_name.get(name, [])] for name in names if len(by_name.get(name, [])) != 1}
    if wrong:
        raise RuntimeError(f"expected exactly one run.json per run under /kaggle/input; found {wrong}. Attach the runner "
                           "output with the sampling predictions and the one with predictions/baseline. "
                           f"/kaggle/input holds:\n{input_overview()}")
    return {name: by_name[name][0] for name in names}


if RUN_SAMPLING_EVAL:
    SAMPLING_RUNS = find_runs(SAMPLING_RUN_NAMES)
    for name, run_dir in SAMPLING_RUNS.items():
        record = json.loads((run_dir / "run.json").read_text())
        print(f"{name}: {run_dir} ({record['config']['inference']}, subsets {record.get('subsets')})")
if SINGLE_RUN:
    if PREDICTIONS_DIR is None:
        runs = find_input_files("run.json")
        if len(runs) != 1:
            raise RuntimeError(f"expected one run.json under /kaggle/input, found {[str(r) for r in runs]}. Attach the "
                               "output of the runner notebook version that wrote predictions/<run name>/run.json, or set "
                               f"PREDICTIONS_DIR in the first cell. /kaggle/input holds:\n{input_overview()}")
        RUN_DIR = runs[0].parent
    else:
        RUN_DIR = Path(PREDICTIONS_DIR)
    run_record = json.loads((RUN_DIR / "run.json").read_text())
    RUN_NAME = run_record["name"]
    EVAL_DIR = Path("/kaggle/working/eval") / RUN_NAME
    CONSISTENCY_DIR = Path("/kaggle/working/consistency") / RUN_NAME
    print(f"\npredictions {RUN_DIR} (run {RUN_NAME!r}, {run_record.get('start_time')} -> {run_record.get('end_time')})")
    for name, seq in run_record["sequences"].items():
        print(f"  {name}: window {seq['window']}, {len(list((RUN_DIR / name).glob('*.npz')))} npz files")
    print(f"results -> {EVAL_DIR}")


In [ ]:
SAMPLING_DIR = Path("/kaggle/working/sampling")
SUBSETS = f"{PROJECT_DIR}/configs/sampling_subsets.yaml"
SAMPLING_CONFIGS = ("zeros_rerun", "noise_seed0", "noise_seed1", "noise_seed2")


def run_parallel(*commands: list[object]) -> None:
    """Run project scripts concurrently, prefixing each output line with its index; fail if any fails."""
    import threading

    procs = [subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                              env={**os.environ, "PYTHONUNBUFFERED": "1"},
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT) for cmd in commands]

    def relay(k: int, proc: subprocess.Popen) -> None:
        for line in proc.stdout:
            print(f"[{k}] {line}", end="", flush=True)

    threads = [threading.Thread(target=relay, args=(k, p), daemon=True) for k, p in enumerate(procs)]
    for thread in threads:
        thread.start()
    codes = [p.wait() for p in procs]
    for thread in threads:
        thread.join()
    if any(codes):
        raise RuntimeError(f"exit codes {codes} for {[str(c[0]) for c in commands]}; see the output above")


if RUN_SAMPLING_EVAL:
    common = ["--data-root", DATA_ROOT, "--sequences-config", SEQUENCES_CONFIG, "--subsets", SUBSETS]
    run(f"{PROJECT_DIR}/scripts/check_determinism.py", "--predictions", SAMPLING_RUNS["zeros_rerun"],
        "--reference", SAMPLING_RUNS["baseline"], "--subsets", SUBSETS, "--out", SAMPLING_DIR / "determinism.csv")
    for name in SAMPLING_CONFIGS:
        run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", SAMPLING_RUNS[name], *common,
            "--out", SAMPLING_DIR / "eval" / name, "--workers", 4)
    # consistency_eval.py runs one worker per sequence and the subsets hold two, so two configs at a time use 4 CPUs.
    for batch in (SAMPLING_CONFIGS[:2], SAMPLING_CONFIGS[2:]):
        run_parallel(*[[f"{PROJECT_DIR}/scripts/consistency_eval.py", "--predictions", SAMPLING_RUNS[name], *common,
                        "--out", SAMPLING_DIR / "consistency" / name, "--workers", 2, "--threads", 1]
                       for name in batch])
    run(f"{PROJECT_DIR}/scripts/sampling_compare.py", "--noise", *(SAMPLING_RUNS[n] for n in SAMPLING_CONFIGS[1:]),
        "--zeros", SAMPLING_RUNS["zeros_rerun"], "--eval-root", SAMPLING_DIR / "eval",
        "--consistency-root", SAMPLING_DIR / "consistency",
        "--baseline-consistency", f"{PROJECT_DIR}/results/consistency/baseline", *common,
        "--out", SAMPLING_DIR, "--workers", 4, "--threads", 1)
    print((SAMPLING_DIR / "summary.md").read_text())
    print("to download:")
    for path in sorted(SAMPLING_DIR.rglob("*")):
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_SAMPLING_EVAL is False; skipping the sampling comparison")


In [ ]:
SWEEP_EVAL_DIR = Path("/kaggle/working/sweep")
SWEEP_SUBSETS = f"{PROJECT_DIR}/configs/sweep_subsets.yaml"
SWEEP_CHECK = "sweep_baseline_check"
# sweep_accuracy.py exits 3 when only its final regression check fails, after sweep_accuracy.csv is written.
REGRESSION_MISMATCH = 3


def find_sweep_runs() -> dict[str, Path]:
    """sweep_* run folders under /kaggle/input by run.json, skipping copies inside an attached project clone."""
    by_name: dict[str, list[Path]] = {}
    for path in find_input_files("run.json"):
        if "/Pointdit-In-Motion/" in str(path):
            continue
        name = json.loads(path.read_text()).get("name", "")
        if name.startswith("sweep_"):
            by_name.setdefault(name, []).append(path.parent)
    several = {name: [str(p) for p in paths] for name, paths in by_name.items() if len(paths) > 1}
    if several or not by_name:
        raise RuntimeError(f"expected one run.json per sweep run under /kaggle/input, found {several or 'none'}. "
                           f"Attach the runner output with predictions/sweep. /kaggle/input holds:\n{input_overview()}")
    return {name: paths[0] for name, paths in sorted(by_name.items())}


def sweep_status(run_dir: Path) -> str:
    path = run_dir / "sweep_status.json"
    return json.loads(path.read_text()).get("status", "unknown") if path.is_file() else "no sweep_status.json"


def run_code(*cmd: object) -> int:
    """Like run(), but returns the exit code instead of raising."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


if RUN_SWEEP_EVAL:
    sweep_runs = find_sweep_runs()
    statuses = {name: sweep_status(run_dir) for name, run_dir in sweep_runs.items()}
    for name, run_dir in sweep_runs.items():
        print(f"{name}: {statuses[name]} ({run_dir})")
    succeeded = [name for name, status in statuses.items() if status == "ok"]
    if SWEEP_CHECK not in succeeded:
        raise RuntimeError(f"{SWEEP_CHECK} is not among the successful sweep runs {succeeded}; it is the reference")
    # Each config is evaluated even if another fails; failures are raised only after every output is written.
    failed = {}
    for name in succeeded:
        code = run_code(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", sweep_runs[name], "--data-root", DATA_ROOT,
                        "--sequences-config", SEQUENCES_CONFIG, "--subsets", SWEEP_SUBSETS,
                        "--out", SWEEP_EVAL_DIR / "eval" / name, "--workers", 4)
        if code:
            failed[name] = code
    code = run_code(f"{PROJECT_DIR}/scripts/sweep_accuracy.py", "--eval-root", SWEEP_EVAL_DIR / "eval",
                    "--configs", *(n for n in succeeded if n not in failed), "--subsets", SWEEP_SUBSETS,
                    "--baseline-csv", f"{PROJECT_DIR}/results/eval/baseline/per_frame.csv",
                    "--out", SWEEP_EVAL_DIR / "sweep_accuracy.csv")
    print("to download:")
    for path in sorted(p for p in SWEEP_EVAL_DIR.rglob("*") if p.is_file()):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    if code == REGRESSION_MISMATCH:
        print(f"\nWARNING the regression check of {SWEEP_CHECK} failed (see the banner above); outputs are kept")
    elif code:
        raise RuntimeError(f"sweep_accuracy.py exited with code {code}; see the output above")
    if failed:
        raise RuntimeError(f"evaluate.py failed for {failed} (run name and exit code); the other configs are tabulated")
else:
    print("RUN_SWEEP_EVAL is False; skipping the sweep evaluation")

In [ ]:
if RUN_CONSISTENCY:
    run(f"{PROJECT_DIR}/scripts/consistency_eval.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--out", CONSISTENCY_DIR, "--workers", 4, "--threads", 1,
        "--save-maps", "rgbd_dataset_freiburg1_360", "rgbd_dataset_freiburg1_desk")
else:
    print("RUN_CONSISTENCY is False; skipping consistency_eval.py")


In [ ]:
PERSISTENCE_DIR = Path("/kaggle/working/persistence")
if RUN_PERSISTENCE:
    run(f"{PROJECT_DIR}/scripts/persistence_eval.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--out", PERSISTENCE_DIR, "--workers", 4, "--threads", 1,
        "--reference", f"{PROJECT_DIR}/results/consistency/baseline")
    print("to download:")
    for path in sorted(PERSISTENCE_DIR.glob("*")):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_PERSISTENCE is False; skipping persistence_eval.py")


In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    def attached_per_frame_csvs() -> list[Path]:
        """per_frame.csv files of attached inputs only, or exactly PER_FRAME_CSV_OVERRIDE when it is set; never one
        this session wrote to /kaggle/working."""
        working = Path("/kaggle/working").resolve()
        if PER_FRAME_CSV_OVERRIDE is not None:
            override = Path(PER_FRAME_CSV_OVERRIDE)
            if not override.is_file():
                raise FileNotFoundError(f"PER_FRAME_CSV_OVERRIDE {override} does not exist")
            print(f"using PER_FRAME_CSV_OVERRIDE {override}")
            found = [override]
        else:
            found = find_input_files("per_frame.csv")
        written_here = [p for p in found if p.resolve().is_relative_to(working)]
        if written_here:
            raise RuntimeError(f"{[str(p) for p in written_here]} under /kaggle/input resolve into /kaggle/working")
        return found


    def check_evaluates_this_run(csv_path: Path) -> None:
        eval_json = csv_path.parent / "eval.json"
        if not eval_json.is_file():
            raise RuntimeError(f"{eval_json} not found next to {csv_path}; cannot tell which run it evaluated")
        evaluated_run = json.loads(eval_json.read_text())["predictions"]["run"]
        if evaluated_run != run_record:
            raise RuntimeError(f"{csv_path} evaluates run {evaluated_run.get('name')!r} started "
                               f"{evaluated_run.get('start_time')}, but the attached predictions are run {RUN_NAME!r} "
                               f"started {run_record.get('start_time')}")


    # Look for the attached CSV before evaluating, so the new CSV cannot be among the candidates.
    attached = attached_per_frame_csvs()
    if RUN_EVALUATION:
        if len(attached) > 1:
            raise RuntimeError(f"found several per_frame.csv under /kaggle/input: {[str(p) for p in attached]}; "
                               "attach at most one older evaluation to compare against")
        for old_csv in attached:
            check_evaluates_this_run(old_csv)
        run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--out", EVAL_DIR, "--workers", WORKERS)
        CSV_PATH = EVAL_DIR / "per_frame.csv"
        if attached:
            print(f"\ncomparing {CSV_PATH} with {attached[0]}")
            run(f"{PROJECT_DIR}/scripts/compare_eval_csv.py", CSV_PATH, attached[0])
        else:
            print("\nno older per_frame.csv attached under /kaggle/input; skipping the reproduction check")
    else:
        if len(attached) != 1:
            raise RuntimeError(f"RUN_EVALUATION is False, so exactly one per_frame.csv must be attached under /kaggle/input; "
                               f"found {[str(p) for p in attached]}. Attach the output of the evaluation notebook version "
                               "that evaluated this run, or set RUN_EVALUATION = True.")
        CSV_PATH = attached[0]
        check_evaluates_this_run(CSV_PATH)
    print(f"per-frame results {CSV_PATH}")


In [ ]:
if RUN_FOCAL_SPLIT:
    run(f"{PROJECT_DIR}/scripts/focal_reliability.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--out", EVAL_DIR / "focal_split.csv")
else:
    print("RUN_FOCAL_SPLIT is False; skipping focal_reliability.py")


In [ ]:
FUSION_DIR = Path("/kaggle/working/fusion")
# "ladder" is the original configuration grid, "edge" the edge-aware options, written to oracle_eval_edge/.
FUSION_CONFIG_GROUP = "ladder"
# None aligns every frame (about an hour). A glob pattern such as "/kaggle/input/*/fusion/oracle_aligned" matching
# exactly one attached output reuses its oracle_aligned folder; fusion_prepare.py then skips the cached frames.
FUSION_CACHE_INPUT = None
if RUN_FUSION:
    import glob
    import shutil

    # ORACLE scale. Every source is aligned to its own GT before fusion, so these results are an upper bound.
    FUSION_CACHE = FUSION_DIR / "oracle_aligned" / RUN_NAME
    eval_folder = "oracle_eval" if FUSION_CONFIG_GROUP == "ladder" else f"oracle_eval_{FUSION_CONFIG_GROUP}"
    FUSION_OUT = FUSION_DIR / eval_folder / RUN_NAME
    if FUSION_CACHE_INPUT:
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        cache_input = Path(matches[0])
        if cache_input.name != "oracle_aligned":
            cache_input = cache_input / "fusion" / "oracle_aligned"
        assert (cache_input / RUN_NAME / "oracle_prepare.json").is_file(), f"{cache_input} has no {RUN_NAME} cache"
        print(f"copying {cache_input} to {FUSION_DIR / 'oracle_aligned'}")
        shutil.copytree(cache_input, FUSION_DIR / "oracle_aligned", dirs_exist_ok=True,
                        copy_function=shutil.copyfile)
    fusion_common = ["--data-root", DATA_ROOT, "--sequences-config", SEQUENCES_CONFIG]
    run(f"{PROJECT_DIR}/scripts/fusion_prepare.py", "--predictions", RUN_DIR, *fusion_common, "--out", FUSION_CACHE,
        "--workers", 4, "--threads", 1)
    fusion_eval_args = ["--cache", FUSION_CACHE, *fusion_common, "--baseline-csv", CSV_PATH, "--workers", 4,
                        "--threads", 1, "--config-group", FUSION_CONFIG_GROUP]
    # None evaluates every sequence; the summary covers every oracle_fusion_<sequence>.csv in FUSION_OUT.
    FUSION_SEQUENCES = None
    fusion_sequences = ["--sequences", *FUSION_SEQUENCES] if FUSION_SEQUENCES else []
    fusion_eval = f"{PROJECT_DIR}/scripts/fusion_eval.py"
    run(fusion_eval, *fusion_eval_args, *fusion_sequences, "--out", FUSION_OUT, "--preflight")
    # Exit code 3 means every output was written but baseline_self differs from per_frame.csv.
    if run(fusion_eval, *fusion_eval_args, *fusion_sequences, "--out", FUSION_OUT, allowed=(0, 3)) == 3:
        print("WARNING fusion_eval.py baseline_self regression check FAILED; outputs are kept, see below")
    print((FUSION_OUT / "regression_status.json").read_text())
    if FUSION_ALL_TARGETS_CONFIG:
        run(fusion_eval, *fusion_eval_args, "--all-targets",
            "--configs", FUSION_ALL_TARGETS_CONFIG,
            "--out", FUSION_DIR / f"{eval_folder}_all_targets" / RUN_NAME)
    print("to download (the oracle_aligned cache is not needed):")
    for path in sorted(FUSION_DIR.rglob("*")):
        if path.is_file() and path.relative_to(FUSION_DIR).parts[0] != "oracle_aligned":
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    print(f"  {FUSION_CACHE / 'oracle_prepare.json'} (provenance of the cache)")
else:
    print("RUN_FUSION is False; skipping fusion_prepare.py and fusion_eval.py")


In [ ]:
if RUN_FUSION_JOINT:
    import glob

    from pim.fusion.oracle import resolve_cache_dir

    # Scale joint_solved, poses oracle_gt. Per frame scale and shift are solved from parallax with no GT depth, the
    # GT poses stay an ORACLE, and the GT-aligned cache is read only for the logging-only comparison of affines.
    if RUN_FUSION:
        joint_cache = FUSION_DIR / "oracle_aligned" / RUN_NAME
    else:
        assert FUSION_CACHE_INPUT, "RUN_FUSION_JOINT needs FUSION_CACHE_INPUT (or RUN_FUSION = True) for the cache"
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        # The notebook output root, its oracle_aligned folder or the oracle_aligned/<run name> cache itself.
        joint_cache = resolve_cache_dir(Path(matches[0]), RUN_NAME)
    assert (joint_cache / "oracle_prepare.json").is_file(), f"{joint_cache} has no oracle_prepare.json"
    # The oracle edge summary of this session if RUN_FUSION wrote one, otherwise the one bundled with the project.
    joint_oracle_summary = FUSION_DIR / "oracle_eval_edge" / RUN_NAME / "oracle_fusion_summary.csv"
    if not joint_oracle_summary.is_file():
        joint_oracle_summary = Path(PROJECT_DIR) / "results" / "fusion" / "oracle_eval_edge" / RUN_NAME / \
            "oracle_fusion_summary.csv"
    JOINT_OUT = FUSION_DIR / "joint_eval" / RUN_NAME
    # None evaluates every sequence of the cache.
    JOINT_SEQUENCES = None
    joint_args = ["--predictions", RUN_DIR, "--cache", joint_cache, "--data-root", DATA_ROOT,
                  "--sequences-config", SEQUENCES_CONFIG, "--baseline-csv", CSV_PATH,
                  "--oracle-summary", joint_oracle_summary, "--workers", 4, "--threads", 1, "--out", JOINT_OUT,
                  *(["--sequences", *JOINT_SEQUENCES] if JOINT_SEQUENCES else [])]
    fusion_joint = f"{PROJECT_DIR}/scripts/fusion_joint.py"
    run(fusion_joint, *joint_args, "--preflight")
    # Exit code 3 means every output was written but baseline_self or the no-fusion control differs.
    if run(fusion_joint, *joint_args, allowed=(0, 3)) == 3:
        print("WARNING fusion_joint.py regression check FAILED; outputs are kept, see below")
    print((JOINT_OUT / "regression_status.json").read_text())
    print("to download")
    for path in sorted(JOINT_OUT.rglob("*")):
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
else:
    print("RUN_FUSION_JOINT is False; skipping fusion_joint.py")


In [ ]:
if RUN_RENDER_PREPARE:
    import glob

    from pim.fusion.oracle import resolve_cache_dir

    # Scale joint_solved, poses oracle_gt. Fused depth of every window frame by joint_causal_adjacent_w4, for videos
    # and 3D maps. The cache is read only for its provenance and windows.
    if RUN_FUSION:
        render_cache = FUSION_DIR / "oracle_aligned" / RUN_NAME
    else:
        assert FUSION_CACHE_INPUT, "RUN_RENDER_PREPARE needs FUSION_CACHE_INPUT (or RUN_FUSION = True) for the cache"
        matches = sorted(glob.glob(FUSION_CACHE_INPUT))
        assert len(matches) == 1, f"FUSION_CACHE_INPUT {FUSION_CACHE_INPUT} must match one path, got {matches}"
        render_cache = resolve_cache_dir(Path(matches[0]), RUN_NAME)
    # The joint results of this session if RUN_FUSION_JOINT wrote them, otherwise the ones bundled with the project.
    render_joint = FUSION_DIR / "joint_eval" / RUN_NAME
    if not (render_joint / "regression_status.json").is_file():
        render_joint = Path(PROJECT_DIR) / "results" / "fusion" / "joint_eval" / RUN_NAME
    RENDER_DIR = FUSION_DIR / "render"
    RENDER_OUT = RENDER_DIR / RUN_NAME
    render_args = ["--predictions", RUN_DIR, "--cache", render_cache, "--data-root", DATA_ROOT,
                   "--sequences-config", SEQUENCES_CONFIG, "--baseline-csv", CSV_PATH, "--joint-csv", render_joint,
                   "--workers", 4, "--threads", 1, "--out", RENDER_DIR,
                   *(["--sequences", *RENDER_SEQUENCES] if RENDER_SEQUENCES else [])]
    render_prepare = f"{PROJECT_DIR}/scripts/fusion_render_prepare.py"
    run(render_prepare, *render_args, "--preflight")
    # Exit code 3 means every output was written but rel_d_raw or rel_d_fused differs from its reference.
    if run(render_prepare, *render_args, allowed=(0, 3)) == 3:
        print("WARNING fusion_render_prepare.py regression check FAILED; outputs are kept, see below")
    print((RENDER_OUT / "regression_status.json").read_text())
    print("to download")
    for path in sorted(RENDER_OUT.glob("*.json")):
        print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    for folder in sorted(p for p in RENDER_OUT.iterdir() if p.is_dir()):
        files = list(folder.glob("*.npz"))
        print(f"  {folder}/ ({len(files)} npz, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB)")
else:
    print("RUN_RENDER_PREPARE is False; skipping fusion_render_prepare.py")


In [ ]:
VIDEO_DIR = Path("/kaggle/working/videos")
if RUN_RENDER_VIDEO or RENDER_STORY or RENDER_WORLD:
    import glob

    # The render preparation output that RUN_RENDER_VIDEO, RENDER_STORY and RENDER_WORLD read.

    def render_frames_dir(path: Path) -> Path:
        """The folder holding the sequence folders of npz frames, given it or a notebook output root."""
        if any(next(p.glob("*.npz"), None) for p in path.iterdir() if p.is_dir()):
            return path
        candidate = path / "fusion" / "render" / RUN_NAME
        if candidate.is_dir():
            return candidate
        raise FileNotFoundError(f"no sequence folders of npz frames in {path} or in {candidate}")

    if RENDER_INPUT is None:
        assert RUN_RENDER_PREPARE, ("RUN_RENDER_VIDEO, RENDER_STORY and RENDER_WORLD need RENDER_INPUT (or "
                                    "RUN_RENDER_PREPARE = True)")
        video_input = RENDER_OUT
    else:
        matches = sorted(glob.glob(str(RENDER_INPUT)))
        assert len(matches) == 1, f"RENDER_INPUT {RENDER_INPUT} must match one path, got {matches}"
        video_input = render_frames_dir(Path(matches[0]))
    video_sequences = RENDER_SEQUENCES or sorted(p.name for p in video_input.iterdir() if p.is_dir())
    print(f"render input {video_input}, sequences {video_sequences}")

In [ ]:
if RUN_RENDER_VIDEO:
    # Scale joint_solved, poses oracle_gt. Videos and point maps of the render preparation output, an ORACLE upper
    # bound. CPU only, one process at a time.

    for name in video_sequences:
        run(f"{PROJECT_DIR}/scripts/render_video.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name, VIDEO_DIR)
    map_name = video_sequences[0]
    MAP_DIR = VIDEO_DIR / f"{map_name}_map"
    run(f"{PROJECT_DIR}/scripts/export_fused_map.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, map_name, MAP_DIR)
    run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, VIDEO_DIR / f"{map_name}_orbit")
    if RENDER_FLYTHROUGH:
        run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, VIDEO_DIR / f"{map_name}_orbit", "--flythrough")
    print(json.dumps(json.loads((MAP_DIR / "map.json").read_text())["ghosting"], indent=2))
    print("to download")
    frame_folders: dict[Path, list[Path]] = {}
    for path in sorted(VIDEO_DIR.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix == ".png" and path.stem.isdigit():
            frame_folders.setdefault(path.parent, []).append(path)
        else:
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder, files in frame_folders.items():
        print(f"  {folder}/ ({len(files)} PNG frames without ffmpeg, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB)")
else:
    print("RUN_RENDER_VIDEO is False; skipping render_video.py, export_fused_map.py and render_orbit.py")


In [ ]:
if RENDER_STORY:
    # Scale joint_solved, poses oracle_gt. Short, slower clips of the render preparation output, an ORACLE upper
    # bound. CPU only, one process at a time.
    STORY_DIR = VIDEO_DIR / "story"
    for name in video_sequences:
        for mode in STORY_MODES:
            run(f"{PROJECT_DIR}/scripts/render_story.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name, STORY_DIR,
                "--mode", mode, *STORY_ARGS.get(mode, []))
    map_name = video_sequences[0]
    MAP_DIR = VIDEO_DIR / f"{map_name}_map"
    if not (MAP_DIR / "map.json").is_file():
        run(f"{PROJECT_DIR}/scripts/export_fused_map.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, map_name, MAP_DIR)
    run(f"{PROJECT_DIR}/scripts/render_orbit.py", MAP_DIR, STORY_DIR / f"{map_name}_orbit", "--side-by-side",
        "--fps", 10, "--seconds", 36, "--hold-seconds", 1)
    for path in sorted(STORY_DIR.glob("*.json")):
        record = json.loads(path.read_text())
        medians = record["clip_medians"]
        frame = record.get("frame", {}).get("index")
        print(f"{path.name}  frames {record['clip']['frames']}  disagreement raw "
              f"{medians['median_disagreement_raw_pct']} fused {medians['median_disagreement_fused_pct']} %"
              + (f"  frame {frame} ({record['frame']['chosen_by']})" if frame is not None else ""))
    print("to download")
    for path in sorted(STORY_DIR.rglob("*")):
        if path.is_file() and not (path.suffix == ".png" and path.stem.isdigit()):
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder in sorted({p.parent for p in STORY_DIR.rglob("[0-9]*.png")}):
        print(f"  {folder}/ (PNG frames without ffmpeg)")
else:
    print("RENDER_STORY is False; skipping render_story.py and the slowed orbit")

In [ ]:
if RENDER_WORLD:
    # Scale joint_solved, poses oracle_gt. World frame point maps of the render preparation output, an ORACLE upper
    # bound and not a dynamic 4D reconstruction. CPU only, one process at a time.
    WORLD_DIR = VIDEO_DIR / "world"
    for name in video_sequences:
        for mode in WORLD_MODES:
            run(f"{PROJECT_DIR}/scripts/render_world_video.py", video_input, DATA_ROOT, SEQUENCES_CONFIG, name,
                WORLD_DIR, "--mode", mode, *WORLD_ARGS)
    for path in sorted(WORLD_DIR.glob("*.json")):
        record = json.loads(path.read_text())
        ghost = {s: g["std_m"] for s, g in record["ghosting"].items()}
        print(f"{path.name}  frames {len(record['frames_used'])}  final points {record['points']['final_shown']['raw']}"
              f"  ghosting std m {ghost}  peak RSS {record['peak_rss_mb']} MB")
    print("to download")
    for path in sorted(WORLD_DIR.rglob("*")):
        if path.is_file() and not (path.suffix == ".png" and path.stem.isdigit()):
            print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")
    for folder in sorted({p.parent for p in WORLD_DIR.rglob("[0-9]*.png")}):
        print(f"  {folder}/ (PNG frames without ffmpeg)")
else:
    print("RENDER_WORLD is False; skipping render_world_video.py")

In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    with open(CSV_PATH, newline="") as f:
        rows = list(csv.DictReader(f))
    evaluated = list(dict.fromkeys(r["sequence"] for r in rows))

    for name in evaluated:
        window = run_record["sequences"][name]["window"]
        middle = window["start"] + window["count"] // 2
        run(f"{PROJECT_DIR}/scripts/check_frame.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--sequence", name, "--frame", middle)
        print()


In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    import numpy as np

    WORST = 5
    ROOM = "rgbd_dataset_freiburg1_room"
    ROOM_FRAMES = range(366, 381)
    PREVIEW_DIR = Path("/kaggle/working/previews") / RUN_NAME
    BREAKDOWN_CSV = PREVIEW_DIR / "error_breakdown.csv"

    if ROOM not in evaluated:
        raise RuntimeError(f"{CSV_PATH} has no {ROOM} rows; evaluated sequences are {evaluated}")


    def preview_frames(name: str) -> dict[int, str]:
        """Frame index -> why it is previewed."""
        scored = sorted((r for r in rows if r["sequence"] == name and r["rel_d"]), key=lambda r: float(r["rel_d"]))
        if not scored:
            raise RuntimeError(f"{name}: no frame with a rel_d in {CSV_PATH}")
        median = float(np.median([float(r["rel_d"]) for r in scored]))
        closest = min(scored, key=lambda r: abs(float(r["rel_d"]) - median))
        reasons: dict[int, list[str]] = {}
        for rank, r in enumerate(reversed(scored[-WORST:]), start=1):
            reasons.setdefault(int(r["frame_index"]), []).append(f"worst{rank}")
        reasons.setdefault(int(closest["frame_index"]), []).append("median")
        if name == ROOM:
            for index in ROOM_FRAMES:
                reasons.setdefault(index, []).append("room366-380")
        return {index: "+".join(why) for index, why in sorted(reasons.items())}


    # The script merges into an existing error_breakdown.csv; start clean so it holds exactly this frame list.
    BREAKDOWN_CSV.unlink(missing_ok=True)
    reasons = {}
    for name in evaluated:
        frames = preview_frames(name)
        reasons.update({(name, index): why for index, why in frames.items()})
        print(name, frames)
        run(f"{PROJECT_DIR}/scripts/preview_predictions.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
            "--sequences-config", SEQUENCES_CONFIG, "--sequence", name, "--frames", *frames,
            "--csv", CSV_PATH, "--out", PREVIEW_DIR)
        print()


In [ ]:
if not SINGLE_RUN:
    print("RUN_SAMPLING_EVAL or RUN_SWEEP_EVAL without PREDICTIONS_DIR; skipping the single-run steps")
else:
    with open(BREAKDOWN_CSV, newline="") as f:
        breakdown = list(csv.DictReader(f))


    def number(value: str, digits: int) -> str:
        return f"{float(value):.{digits}f}" if value else "n/a"


    print("band_px / band_err: share of valid pixels / of total relative error within 5 px of a GT depth edge or hole.")
    print("band_err >> band_px: error sits on edges; band_err ~ band_px with high rel_out: error spread over regions.")
    for name in evaluated:
        print(f"\n{name}")
        print(f"{'frame':>6} {'rel_d':>7} {'band_px':>8} {'band_err':>8} {'ratio':>6} {'rel_band':>8} {'rel_out':>8} "
              f"{'offset_ms':>9}  why")
        for r in (r for r in breakdown if r["sequence"] == name):
            ratio = (f"{float(r['band_error_fraction']) / float(r['band_pixel_fraction']):.2f}"
                     if r["band_error_fraction"] and float(r["band_pixel_fraction"]) > 0 else "n/a")
            print(f"{r['frame_index']:>6} {number(r['rel_d'], 2):>7} {number(r['band_pixel_fraction'], 3):>8} "
                  f"{number(r['band_error_fraction'], 3):>8} {ratio:>6} {number(r['rel_d_band'], 2):>8} "
                  f"{number(r['rel_d_outside'], 2):>8} {number(r['depth_offset_ms'], 1):>9}  "
                  f"{reasons[name, int(r['frame_index'])]}")

    print("\nto download:")
    for directory in (EVAL_DIR, PREVIEW_DIR, CONSISTENCY_DIR):
        for path in sorted(directory.rglob("*")) if directory.is_dir() else []:
            if path.is_file() and "maps" not in path.relative_to(directory).parts:
                print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
    for maps in sorted((CONSISTENCY_DIR / "maps").glob("*")) if (CONSISTENCY_DIR / "maps").is_dir() else []:
        print(f"  {maps}/ ({len(list(maps.glob('*.npz')))} npz, "
              f"{sum(p.stat().st_size for p in maps.glob('*.npz')) / 1e6:.0f} MB)")
